Evaluate QML model:
* 6 input feature
* Shots used during training:  no_shots_training = "inf' or 1000
* Shots used during inference: no_shots = "inf" or integer
* architecture name_arch = "ZZXY", "XYZ"
* evaluated on first 1000 points (where QML is called, i.e. (clt <= 1.0e-08) ) of each cloud regime

MSE and R2 score are saved in 'qml_reference_' + name_arch + '_st'+str(no_shots_training) + 'no_varreg' + '_s' +str(no_shots)+'N1000'

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
from pathlib import Path
import pennylane as qml
import pennylane.numpy as np
import jax
from jax import numpy as jnp
from sklearn.metrics import r2_score,mean_squared_error



jax.config.update("jax_enable_x64", True)

path_base = Path(os.getcwd()) 
# Current path for importing custom functions
sys.path.insert(0, str(path_base / "clc_functions")) 


import input_transform
importlib.reload(input_transform)
from input_transform import inputs_transform, inverse_transform_clc



import qnn_layouts_pennylane
importlib.reload(qnn_layouts_pennylane)
import qnn_layouts_pennylane as pqcs

In [ ]:

# Folder in which to find the test inputs
test_inputs_folder = 'test_data/'


transform_output = True
name_out_transf = ''
if transform_output:
    name_out_transf = '_transformedCLC'

### Interval within which transformed clc should be bounded
bound_output = [0.0, 1.0]

### Upper bound for input transformation
transform_input = True
upperbound = np.pi
upperbound_name = '1p0pi'

batch_size = 100
n_batch_name = str(batch_size)

# Learning rate
learning_rate = 0.001
learning_rate_name = '0p001'

### Kept features
features_kept = ['hus', 'clw', 'cli', 'ta', 'pa', 'hwind']
no_of_features = len(features_kept)

### Architecture specifications
no_qubits = no_of_features

name_arch = 'ZZXY'
no_shots = 1000
no_shots_training = 1000#'inf' 
### PQC architecture layout
if name_arch == 'XYZ':
    pqc_layout = pqcs.XYZ_circuit;  name_arch = 'XYZ';  n_enc = 4;  n_dec = 2; 
    if no_shots_training == 'inf':
        best_exp = 1 # for infinite shots
        
    else:
        best_exp = 4#
elif name_arch == 'ZZXY':
    pqc_layout = pqcs.ZZXY_circuit;  name_arch = 'ZZXY';  n_enc = 2;  n_dec = 5;
    if no_shots_training == 'inf':
        best_exp = 6 # for infinite shots
    else:
        best_exp = 3 #varreg


n_enc_name = str(n_enc)
n_dec_name = str(n_dec)
# Load optimal params:
if no_shots_training == 'inf':
    params_folder = 'optimal_params/'
    namefilepars = 'optimal_params'
    name_end = ('_upperbound' + upperbound_name + name_out_transf + '_' + name_arch + '_Nenc' + n_enc_name + 
                '_Ndec' + n_dec_name + '_batch' + n_batch_name + '_lr' + learning_rate_name + '_test' + str(best_exp))
    filename_pars = namefilepars + name_end + '.npy'
else: #varreg, trainign with 1000t
    params_folder = 'optimal_params/'    
    filename_pars = 'optimal_params_Nshots'+str(no_shots_training)+'_multinomial_transformedCLC_' + name_arch + '_Nenc' +str(n_enc) + '_Ndec' + str(n_dec) + '_batch100_alpha0p005_iniparam12345_test' + str(best_exp) +'.npy'
   

path_file = os.path.join(params_folder, filename_pars)
opt_params_np = np.load(path_file)
opt_params = jnp.asarray(opt_params_np)






In [3]:
ind_features = [0,1,2,3,4,6]


In [ ]:

### ---------------------------------------------------------------------------------------- ###
## ----------------------------------- Load testing data ------------------------------------ ##
### ---------------------------------------------------------------------------------------- ###

namefilein = 'cirrus_inputs_raw_8features.npy'
namefileout = 'cirrus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cirrus_full = np.load(path_file)
test_inputs_cirrus = test_inputs_cirrus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cirrus = np.load(path_file)
no_testing_data_cirrus = test_inputs_cirrus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cirrus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cirrus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cirrus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cirrus = np.where(III_cirrus)[0]
no_test_samples_to_evaluate_cirrus = np.sum(III_cirrus)
print('No. test samples to evaluate (cirrus): ', no_test_samples_to_evaluate_cirrus)

namefilein = 'cumulus_inputs_raw_8features.npy'
namefileout = 'cumulus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cumulus_full = np.load(path_file)
test_inputs_cumulus = test_inputs_cumulus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cumulus = np.load(path_file)
no_testing_data_cumulus = test_inputs_cumulus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cumulus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cumulus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cumulus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cumulus = np.where(III_cumulus)[0]
no_test_samples_to_evaluate_cumulus = np.sum(III_cumulus)
print('No. test samples to evaluate (cumulus): ', no_test_samples_to_evaluate_cumulus)

namefilein = 'deepconv_inputs_raw_8features.npy'
namefileout = 'deepconv_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_deepconv_full = np.load(path_file)
test_inputs_deepconv = test_inputs_deepconv_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_deepconv = np.load(path_file)
no_testing_data_deepconv = test_inputs_deepconv.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_deepconv[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_deepconv[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_deepconv = np.squeeze(np.logical_not(IIIlowclt))
indsIII_deepconv = np.where(III_deepconv)[0]
no_test_samples_to_evaluate_deepconv = np.sum(III_deepconv)
print('No. test samples to evaluate (deepconv): ', no_test_samples_to_evaluate_deepconv)

namefilein = 'stratus_inputs_raw_8features.npy'
namefileout = 'stratus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_stratus_full = np.load(path_file)
test_inputs_stratus = test_inputs_stratus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_stratus = np.load(path_file)
no_testing_data_stratus = test_inputs_stratus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_stratus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_stratus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_stratus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_stratus = np.where(III_stratus)[0]
no_test_samples_to_evaluate_stratus = np.sum(III_stratus)
print('No. test samples to evaluate (stratus): ', no_test_samples_to_evaluate_stratus)

### Transform inputs if needed
if transform_input:
    bound_input = [0.0, upperbound]
    bounds = [bound_input for _ in features_kept]
    test_inputs_cirrus_t = inputs_transform(test_inputs_cirrus, features_kept, bounds)
    test_inputs_cumulus_t = inputs_transform(test_inputs_cumulus, features_kept, bounds)
    test_inputs_deepconv_t = inputs_transform(test_inputs_deepconv, features_kept, bounds)
    test_inputs_stratus_t = inputs_transform(test_inputs_stratus, features_kept, bounds)

### Convert testing data to jax numpy arrays
jnp_test_inputs_cirrus = jnp.asarray(test_inputs_cirrus_t)
jnp_test_inputs_cumulus = jnp.asarray(test_inputs_cumulus_t)
jnp_test_inputs_deepconv = jnp.asarray(test_inputs_deepconv_t)
jnp_test_inputs_stratus = jnp.asarray(test_inputs_stratus_t)


In [5]:

### ---------------------------------------------------------------------------------------- ###
## ---------------------------------- Initialize QNN model ---------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

no_gate_angles = pqcs.no_of_angles_pqc(name_arch, no_qubits, n_enc, n_dec)
no_params = no_gate_angles + no_qubits + 1


if no_shots == 'inf':
    dev = qml.device('default.qubit.jax', wires=no_qubits)
else:
    dev = qml.device('default.qubit.jax', wires=no_qubits, shots=no_shots)

### Define pqc with measured observables
@qml.qnode(dev, interface="jax")
def qnn_pqc(inputs, pars):
    pqc_layout(inputs, pars, n_enc=n_enc, n_dec=n_dec, wires=dev.wires)
    return [qml.expval(qml.PauliZ(i)) for i in range(no_qubits)]

### Define the QNN model (pqc + postprocessing)
@jax.jit
def model_qnn(params, inputs):
    # Split paramter vector in the different components 
    no_angles = no_gate_angles
    no_weights = no_qubits
    no_bias = 1
    angles = jax.lax.dynamic_slice(params, [0], [no_angles])
    weights = jax.lax.dynamic_slice(params, [no_angles], [no_weights])
    bias = jax.lax.dynamic_slice(params, [no_angles + no_weights], [no_bias])

    # Computation of the quantum circuit in 'measured_batches'
    # 'measured_batches' contains a no_qubits-long list, 
    # where the i-th element is the jnp.array containing the 
    # Z(i) expectation value over the input batch
    measured_batches = qnn_pqc(inputs, angles)

    # Classical post-processing (weighted avg. + bias)
    weighted_output = weights[0] * measured_batches[0]
    for i in range(1,no_qubits):
        weighted_output = weighted_output + weights[i] * measured_batches[i]
    predictions = weighted_output + bias
    return jnp.squeeze(predictions)



In [6]:

data_n = {'arch': name_arch, 'Path params':  filename_pars, 'shots': no_shots, 'shots_training': no_shots_training}

## Evaluate model and postproccess [1000 datapoints]

## Cirrus clouds

In [ ]:

#Eval cs  + post processing
batch_size = 1000


pred_test_outputs = np.zeros(no_testing_data_cirrus)  
no_batches_test = int(np.floor(no_test_samples_to_evaluate_cirrus / batch_size))  
for kk in range(0,1):
    if (kk%4000 == 0):
        print(kk)
    III_to_eval = indsIII_cirrus[kk*batch_size:(kk+1)*batch_size]  
    ins_batch = jnp_test_inputs_cirrus[III_to_eval, :]  
    
    outs_batch =  model_qnn(opt_params,ins_batch)

    outs_batch = np.asarray(outs_batch)
    outs_batch = np.squeeze(outs_batch)
    # Clip values in [0,1]
    outs_batch = np.minimum(outs_batch, 1.0)
    outs_batch = np.maximum(outs_batch, 0.0)
    # If the output has been transformed, re-transform it back
    if transform_output:
         outs_batch = inverse_transform_clc(outs_batch)
    pred_test_outputs[III_to_eval] = outs_batch


print('done')


In [8]:

mse_cirrus = mean_squared_error(test_outputs_cirrus[indsIII_cirrus[0:1000]],pred_test_outputs[indsIII_cirrus[0:1000]])
r2_cirrus = r2_score(test_outputs_cirrus[indsIII_cirrus[0:1000]],pred_test_outputs[indsIII_cirrus[0:1000]])

## Cumulus clouds

In [ ]:
batch_size = 1000

pred_test_outputs = np.zeros(no_testing_data_cumulus)  
no_batches_test = int(np.floor(no_test_samples_to_evaluate_cumulus / batch_size))  
for kk in range(0,1):
    III_to_eval = indsIII_cumulus[kk*batch_size:(kk+1)*batch_size]  
    ins_batch = jnp_test_inputs_cumulus[III_to_eval, :]  
    outs_batch =  model_qnn(opt_params,ins_batch)

    outs_batch = np.asarray(outs_batch)
    outs_batch = np.squeeze(outs_batch)
    # Clip values in [0,1]
    outs_batch = np.minimum(outs_batch, 1.0)
    outs_batch = np.maximum(outs_batch, 0.0)
    # If the output has been transformed, re-transform it back
    if transform_output: 
        outs_batch = inverse_transform_clc(outs_batch)
    pred_test_outputs[III_to_eval] = outs_batch

print('done')

In [10]:


mse_cumulus = mean_squared_error(test_outputs_cumulus[indsIII_cumulus[0:1000]],pred_test_outputs[indsIII_cumulus[0:1000]])
r2_cumulus = r2_score(test_outputs_cumulus[indsIII_cumulus[0:1000]],pred_test_outputs[indsIII_cumulus[0:1000]])

## Stratus clouds

In [ ]:

pred_test_outputs = np.zeros(no_testing_data_stratus)  
no_batches_test = int(np.floor(no_test_samples_to_evaluate_stratus / batch_size))  
for kk in range(0,1):
    III_to_eval = indsIII_stratus[kk*batch_size:(kk+1)*batch_size]  
    ins_batch = jnp_test_inputs_stratus[III_to_eval, :]  
    outs_batch =  model_qnn(opt_params,ins_batch)

    outs_batch = np.asarray(outs_batch)
    outs_batch = np.squeeze(outs_batch)
    # Clip values in [0,1]
    outs_batch = np.minimum(outs_batch, 1.0)
    outs_batch = np.maximum(outs_batch, 0.0)
    # If the output has been transformed, re-transform it back
    if transform_output: 
        outs_batch = inverse_transform_clc(outs_batch)
    pred_test_outputs[III_to_eval] = outs_batch
    

In [12]:
mse_stratus = mean_squared_error(test_outputs_stratus[indsIII_stratus[0:1000]],pred_test_outputs[indsIII_stratus[0:1000]])
r2_stratus = r2_score(test_outputs_stratus[indsIII_stratus[0:1000]],pred_test_outputs[indsIII_stratus[0:1000]])

In [ ]:
batch_size = 1000
pred_test_outputs = np.zeros(no_testing_data_deepconv)  
no_batches_test = int(np.floor(no_test_samples_to_evaluate_deepconv / batch_size))  
for kk in range(0,1):
    III_to_eval = indsIII_deepconv[kk*batch_size:(kk+1)*batch_size]  
    ins_batch = jnp_test_inputs_deepconv[III_to_eval, :]  
    outs_batch =  model_qnn(opt_params,ins_batch)

    outs_batch = np.asarray(outs_batch)
    outs_batch = np.squeeze(outs_batch)
    # Clip values in [0,1]
    outs_batch = np.minimum(outs_batch, 1.0)
    outs_batch = np.maximum(outs_batch, 0.0)
    # If the output has been transformed, re-transform it back
    if transform_output: 
        outs_batch = inverse_transform_clc(outs_batch)
    pred_test_outputs[III_to_eval] = outs_batch
    

In [14]:
mse_deepconv = mean_squared_error(test_outputs_deepconv[indsIII_deepconv[0:1000]],pred_test_outputs[indsIII_deepconv[0:1000]])
r2_deepconv = r2_score(test_outputs_deepconv[indsIII_deepconv[0:1000]],pred_test_outputs[indsIII_deepconv[0:1000]])

In [ ]:
data_n2 = data_n
data_n2['No datapoints'] = 1000
data_n2['deepconv'] =  [mse_deepconv,r2_deepconv]
data_n2['stratus'] =  [mse_stratus,r2_stratus]
data_n2['cirrus'] =  [mse_cirrus,r2_cirrus]
data_n2['cumulus'] =  [mse_cumulus,r2_cumulus]
print(data_n2)

In [ ]:
name_data = 'qml_reference_' + name_arch + '_st'+str(no_shots_training) + '_s' +str(no_shots)+'N1000'

np.save(name_data,data_n2)